https://arxiv.org/abs/2603.22597

In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

# cross-match Lafarga+2026 PC with GaiaDR3

In [ ]:
import pandas as pd

df_pc = pd.read_csv('../data/raven_targets.csv')
df_pc.head()

## match 3d separation with cluster core

In [ ]:
from astroquery.mast import Catalogs
import numpy as np

# Query the TIC catalog for these IDs
# We specifically want the 'GAIA' column which contains the Gaia DR2/DR3 source_id
tic_data = Catalogs.query_criteria(catalog="TIC", ID=df_pc['tic'].unique())

In [ ]:
df_gaia = tic_data.to_pandas()[['ID','GAIA','ra','dec','plx','pmRA','pmDEC']].astype(float)
df_gaia['tic'] = df_gaia.ID.copy()
df_pc = pd.merge(df_pc, df_gaia, on='tic')
df_pc.shape

In [ ]:
from astroquery.gaia import Gaia

# Extract unique Gaia IDs and remove masked/null values
gaia_ids = [int(gid) for gid in tic_data['GAIA'] if gid is not np.ma.masked]

# Format the IDs for the SQL 'IN' clause
ids_str = ",".join(map(str, gaia_ids))

# ADQL query to get source_id and radial_velocity
query = f"""
SELECT source_id, radial_velocity, radial_velocity_error
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

# Launch the job
job = Gaia.launch_job_async(query)
results = job.get_results()

In [ ]:
df_rv = results.to_pandas().astype(float).rename({
    'SOURCE_ID': 'GAIA',
    }, axis=1
                                  )
df_pc = pd.merge(df_pc, df_rv, on='GAIA')

In [ ]:
df_pc = df_pc.rename({
        "GAIA": "GaiaDR3",
        "pmRA": "pmra",
        "pmDEC": "pmdec",
        "plx": "parallax",
        "RV": "radial_velocity"
        
    }, axis=1)
df_pc.head()

# Perren+2023 catalog

In [ ]:
import pandas as pd

catalog_name = 'Perren2023'
df = pd.read_csv('../data/Perren2023/UCC_cat.csv')
df.tail()

In [ ]:
df = df.rename({"name": "Name",
                #"Source": "GaiaDR3",
                "probs": "Prob",
                "RA_ICRS": "ra",
                "DE_ICRS": "dec",
                "pmRA": "pmra",
                "pmDE": "pmdec",
                "Plx": "parallax",
                "RV": "radial_velocity"
                }, axis=1)

In [ ]:
df_mem = pd.read_parquet('../data/Perren2023/UCC_members.parquet')
df_mem.tail()

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem = df_mem.rename({"name": "Name",
                        "Source": "GaiaDR3",
                        "probs": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem.shape

In [ ]:
idx = df_mem.GaiaDR3.isin(df_pc.GaiaDR3)
idx.sum()

In [ ]:
from astropy.coordinates import SkyCoord, Distance
import astropy.units as u

coords1 = SkyCoord(ra=df_pc['ra'].values*u.degree, 
                   dec=df_pc['dec'].values*u.degree, 
                   distance=Distance(parallax=df_pc['parallax'].values* u.mas, allow_negative=True),
                   frame='icrs')
coords2 = SkyCoord(ra=df['ra'].values*u.degree, 
                   dec=df['dec'].values*u.degree, 
                   distance=Distance(parallax=df['parallax'].values* u.mas, allow_negative=True),
                   frame='icrs') 

idx, d2d, d3d = coords1.match_to_catalog_sky(coords2)

df_pc['sep2d_arcsec'] = d2d.to(u.arcsec).value
df_pc['sep3d_pc'] = d3d.to(u.pc).value
df_pc.sort_values('sep2d_arcsec').head()

In [ ]:
import sys
sys.path.insert(0, '..')
from wakai.catalog import get_cluster_3d_size

# test on praesepe
d = df_mem.groupby('Name').get_group('ic2602')
get_cluster_3d_size(d)

In [ ]:
# apply func to all clusters
df_size = df_mem.groupby('Name').apply(get_cluster_3d_size)

In [ ]:
df = df.merge(df_size.reset_index(name='extent_pc'), on='Name')
df.head()

In [ ]:
# Add a prefix to matched columns to avoid confusion (optional but recommended)
col_prefix = 'matched_'
df_matched_tic = df.iloc[idx].copy().add_prefix(col_prefix)

# Concatenate horizontally
df_match = pd.concat([df_pc.reset_index(drop=True), 
                      df_matched_tic.reset_index(drop=True)
                      ], axis=1)

In [ ]:
# make sure the 3d sep from cluster core is smaller the spatial extent of that cluster
idx2 = df_match[f'{col_prefix}extent_pc'] > df_match['sep3d_pc']
df_match[idx2]

In [ ]:
df_match['pm_diff'] =  np.sqrt((df_match[f'{col_prefix}pmra'] - df_match[f'{col_prefix}pmdec'])**2 + 
                               (df_match['pmra'] - df_match['pmdec'])**2)


In [ ]:
idx3 = df_match[idx2]['pm_diff'] < 10
cols = ['tic','name','matched_Name','matched_Age_[Myr]',
        'matched_extent_pc','sep3d_pc','pm_diff','GaiaDR3']
df_final = df_match[idx2].loc[idx3,cols]
df_final

In [ ]:
from wakai.catalog import plot_xyz_uvw

n = 0
ticid = df_final["tic"].values[n]
age = df_final["matched_Age_[Myr]"].values[n]
cname = df_final["matched_Name"].values[n]

df_target = df_match.query("tic==@ticid").squeeze()
d = df_mem.groupby("Name").get_group(cname)

fig, ax = plot_xyz_uvw(d, 
                       color='Prob', 
                       df_target=df_target,
                       target_color="C3",
                       target_ms=20,
                       marker='.',
                       cmap="Blues");
fig.suptitle(f"TIC-{int(ticid)} in {cname} (age={age:.0f} Myr)")

In [ ]:
import matplotlib.pyplot as pl
from wakai.catalog import plot_rdp_pmrv
from wakai.catalog import plot_xyz_uvw

outdir = '../output/Lafarga26_pc_near_clusters'
groups = df_mem.groupby("Name")
for i,row in df_final.iterrows():
    try:
        ticid = row.tic
        df_target = df_match.query("tic==@ticid").squeeze()
        cname = row["matched_Name"]
        age = row["matched_Age_[Myr]"]
        d = groups.get_group(cname)
        # if len(d.radial_velocity.dropna())<10:
        #     print(f"Skipping TIC {ticid}")
        #     continue
        fp = f'{outdir}/TIC-{ticid}_in_{cname}_{age:.0f}Myr'
        if False:
            target_gaiaid = row['GaiaDR3']
            fig = plot_rdp_pmrv(d,
                               color='Prob', 
                               target_gaiaid=target_gaiaid,
                               match_id=False,
                               df_target=df_target,
                               target_color="C3",
                               marker='.',
                               cmap="Blues");
            fp=fp+'.png'
        else:
            fig, ax = plot_xyz_uvw(d, 
                               color='Prob', 
                               df_target=df_target,
                               target_color="C3",
                               target_ms=20,
                               marker='.',
                               cmap="Blues",
                               verbose=False
                              );
            fp=fp+'_xyzuvw.png'
        fig.suptitle(f"TIC-{ticid} in {cname} ({age:.0f} Myr)")
        print(f"https://ucc.ar/_clusters/{cname}")
        fig.savefig(fp, bbox_inches='tight', dpi=200)
        # pl.show()
    except Exception as e:
        print(e)